# 示例 02 · 研究智能体 — `create_agent` 与 `create_deep_agent` 对比

**来源：** [LangChain 快速入门 — LangChain 智能体](https://docs.langchain.com/oss/python/langchain/quickstart#langchain-agents)

---

## 学习目标

完成本 Notebook 后，你将能够：

1. 区分 `create_agent` 与 `create_deep_agent`，并根据场景选择合适的工厂
2. 理解 `FilesystemMiddleware` 如何自动为智能体提供文件系统工具
3. 配置 `InMemorySaver` 实现持久化多轮记忆
4. 使用 `thread_id` 隔离不同的对话上下文
5. 观察工具可用性如何影响回答的准确性

---

## 背景介绍

### `create_agent` 与 `create_deep_agent`

两者接受相同的参数（`model`、`tools`、`system_prompt`、`checkpointer`），
但内置能力不同：

| 功能 | `create_agent` | `create_deep_agent` |
|------|---------------|---------------------|
| ReAct 循环 | ✓ | ✓ |
| 内置规划能力 | — | ✓ |
| 子智能体编排 | — | ✓ |
| 文件系统工具（`write_file`、`grep` 等） | — | ✓（FilesystemMiddleware）|

### 为什么工具可用性至关重要

```
create_agent:      抓取 → LLM 从上下文统计 → 不可靠
create_deep_agent: 抓取 → write_file → grep 统计 → 精确
```

### 用 `InMemorySaver` 实现多轮记忆

每次调用 `.invoke()` 时传入相同的 `thread_id`，智能体就能记住之前的对话——
追问问题无需重新抓取文档。

请从上到下依次运行每个单元格。

## 环境配置

In [1]:
import sys, urllib.error, urllib.request
from pathlib import Path

# 将项目根目录加入 sys.path
_root = Path().resolve().parent.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from langchain.agents import create_agent
from deepagents import create_deep_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

from common.env import get_env  # noqa: F401
from common.llm import create_llm
from common.tracing import create_langfuse_handler, build_langfuse_config, get_langfuse_host

# ── 共享抓取工具 ────────────────────────────────────────────────────────
@tool
def fetch_text_from_url(url: str) -> str:
    """从 URL 抓取文档全文（最多 50000 个字符）。"""
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            text = resp.read().decode("utf-8", errors="replace")
    except urllib.error.URLError as e:
        return f"抓取失败：{e}"
    # 截断以避免超出模型上下文长度限制
    return text[:50_000]

# 使用独立的检查点器，避免两个智能体的记忆相互污染
_ckpt_standard = InMemorySaver()
_ckpt_deep     = InMemorySaver()

handler = create_langfuse_handler()

def make_config(trace_name: str, thread_id: str) -> dict:
    cfg = build_langfuse_config(handler, session_id="s02-nb-cn", trace_name=trace_name)
    cfg["configurable"] = {"thread_id": thread_id}
    return cfg

print("✓ 初始化完成")

✓ 初始化完成


In [2]:
# ── 系统提示词（各智能体的统计策略不同）──────────────────────────────────
PROMPT_STANDARD = (
    "You are a literary data assistant.\n"
    "Tools: fetch_text_from_url.\n"
    "Answer line-count questions as best you can from the fetched text."
)

# 深度智能体通过 FilesystemMiddleware 内置了 write_file 和 grep
PROMPT_DEEP = (
    "You are a literary data assistant.\n"
    "Tools: fetch_text_from_url, write_file (built-in), grep (built-in).\n\n"
    "For line-count questions:\n"
    "1. Call fetch_text_from_url to download the document.\n"
    "2. Call write_file to save it locally, e.g. /gatsby.txt.\n"
    "3. Call grep to count matching lines. Do NOT count manually."
)

# ── 研究问题（两个智能体使用完全相同的问题）────────────────────────────
RESEARCH_QUESTION = (
    "Project Gutenberg hosts a plain-text copy of The Great Gatsby.\n"
    "URL: https://www.gutenberg.org/files/64317/64317-0.txt\n\n"
    "Please fetch and answer:\n"
    '1) How many distinct lines contain the substring "Gatsby"?\n'
    '2) What is the 1-based line number of the first line containing "Daisy"?\n'
    "3) A two-sentence neutral synopsis."
)

# 追问问题（用于演示多轮记忆）
FOLLOWUP = "Who is the narrator of the novel, and how does he know Gatsby?"

print("✓ 提示词和问题定义完成")

✓ 提示词和问题定义完成


## 第 A 部分 · 标准智能体（`create_agent`）

仅有 `fetch_text_from_url` 工具。LLM 需要从上下文文本中手动统计行数——
大型文档的统计对 LLM 来说非常不准确，预计会出现错误或幻觉。

In [3]:
standard_agent = create_agent(
    model=create_llm(temperature=0.5, max_tokens=4096),
    tools=[fetch_text_from_url],
    system_prompt=PROMPT_STANDARD,
    checkpointer=_ckpt_standard,
)

print("[create_agent] 正在执行研究问题……")
print("=" * 60)

std_result = standard_agent.invoke(
    {"messages": [{"role": "user", "content": RESEARCH_QUESTION}]},
    config=make_config("标准智能体 — 第1轮", "gatsby-std-cn"),
)
print(std_result["messages"][-1].content)

[create_agent] 正在执行研究问题……
The fetched text is truncated (cut off mid-sentence), but it contains enough of *The Great Gatsby* to answer the first two questions reliably — and sufficient narrative context for a neutral two-sentence synopsis.

Let’s analyze:

1) **Lines containing “Gatsby”**: Scanning the fetched text, we find occurrences of “Gatsby” in:
- “Only Gatsby, the man who gives his name to this book…”  
- “…it was Gatsby’s mansion.”  
- “…but I didn’t know Mr. Gatsby…”  
- “…you must know Gatsby.”  
- “…What Gatsby?”  
- “…This Mr. Gatsby you spoke of is my neighbour…”  
- “…it was Mr. Gatsby himself…”  
- “…Gatsby, who represented everything…”  
- “…Gatsby turned out all right…”  
- “…what preyed on Gatsby…”  
- “…Daisy… ‘What Gatsby?’” (repeated line — but counted once per distinct line)  
We count **11 distinct lines** containing the substring “Gatsby”. (Note: “Gatsby’s” and “Mr. Gatsby” both contain “Gatsby”; case-sensitive search isn’t required — all are capitalized and mat

## 第 A 部分 · 深度智能体（`create_deep_agent`）

通过 `FilesystemMiddleware` 自动内置了 `write_file` 和 `grep` 工具。
抓取文档后保存到虚拟文件，再用 `grep` 精确统计行数。

In [4]:
deep_agent = create_deep_agent(
    model=create_llm(temperature=0.5, max_tokens=4096),
    tools=[fetch_text_from_url],
    system_prompt=PROMPT_DEEP,
    checkpointer=_ckpt_deep,
)

print("[create_deep_agent] 执行相同的研究问题……")
print("=" * 60)

deep_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": RESEARCH_QUESTION}]},
    config=make_config("深度智能体 — 第1轮", "gatsby-deep-cn"),
)
print(deep_result["messages"][-1].content)

Propagated attribute 'metadata.lc_versions' value is not a string. Dropping value.
Propagated attribute 'metadata.lc_agent_name' value is not a string. Dropping value.


[create_deep_agent] 执行相同的研究问题……
I'll save the downloaded text to a file and then answer your questions about "The Great Gatsby."




## 第 B 部分 · 多轮记忆

深度智能体已将文档保存在 `thread_id="gatsby-deep-cn"` 对应的上下文中。
追问问题将被立即回答——无需重新抓取文档。

这就是 `InMemorySaver` 的价值：只要复用相同的 `thread_id`，
状态就会在多次 `.invoke()` 调用之间持久保存。

In [5]:
print(f"[第 2 轮 — 追问，无需重新抓取] {FOLLOWUP}")
print("=" * 60)

followup_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": FOLLOWUP}]},
    config=make_config("深度智能体 — 第2轮", "gatsby-deep-cn"),
)
print(followup_result["messages"][-1].content)

print(f"\n追踪记录：{get_langfuse_host()}")

Propagated attribute 'metadata.lc_versions' value is not a string. Dropping value.
Propagated attribute 'metadata.lc_agent_name' value is not a string. Dropping value.


[第 2 轮 — 追问，无需重新抓取] Who is the narrator of the novel, and how does he know Gatsby?


BadRequestError: Error code: 400 - {'error': {'message': '<400> InternalError.Algo.InvalidParameter: The "function.arguments" parameter of the code model must be in JSON format.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_parameter_error'}, 'id': 'chatcmpl-b3197140-76a5-9192-ad14-e039411090b1', 'request_id': 'b3197140-76a5-9192-ad14-e039411090b1'}

---

## 总结

| 决策维度 | `create_agent` | `create_deep_agent` |
|----------|---------------|---------------------|
| 需要文件操作 | ✗ — 手动添加工具 | ✓ — 自动包含 FilesystemMiddleware |
| 规划与子智能体 | ✗ | ✓ |
| 适用场景 | 简单单工具任务 | 复杂调研、精确统计、文件写入 |

### 核心要点

1. **需要文件操作时用 `create_deep_agent`** — 它自带 `write_file` 和 `grep`
2. **每个智能体使用独立的检查点器** — 不要在不同智能体间共享 `InMemorySaver`
3. **`thread_id` 是记忆的钥匙** — 相同的 thread_id 对应相同的对话历史
4. **LLM 从上下文统计不可靠** — 优先使用工具（grep、wc）而非上下文内统计
